## Convergence Comparison: Triton Kernel Patched vs. Original Model Layer-by-Layer

We have learned how to validate the correctness and performance of a single kernel. However, that does not mean the kernels will run smoothly in the production. In the real production training, the contiguity, the tensor shape, and the dtype might be different than what you have used in the unit tests. We can mimic the real production training but in much smaller scale to validate the result, logits, weights, loss, etc are exact


### Ensure you are using GPU by running `nvidia-smi`

In [ ]:
!nvidia-smi

Wed Aug 21 17:42:03 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   34C    P8               9W /  70W |      0MiB / 15360MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

### Install dependencies

In [ ]:
!pip install transformers datasets liger_kernel "huggingface_hub[cli]"

  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_nccl_cu12-2.20.5-py3-none-manylinux2014_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_nvtx_cu12-12.1.105-py3-none-manylinu

### Login Hugging Face for the tokenizer

In [ ]:
!huggingface-cli login --token <token>

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: read).
Your token has been saved to /root/.cache/huggingface/token
Login successful


### Load a small dataset, tokenizer, and create a small model

In [ ]:
# 1. Load the dataset
from datasets import load_dataset

dataset = load_dataset('tiny_shakespeare')["train"]


# 2. Load the tokenizer

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.2",
)
tokenizer.pad_token = tokenizer.eos_token


# 3. Create a mini model

from transformers import MistralConfig, MistralForCausalLM

model_config = MistralConfig(
    attention_dropout=0.0,
    bos_token_id=1,
    eos_token_id=2,
    hidden_act="silu",
    hidden_size=1024,  # 4096
    initializer_range=0.02,
    intermediate_size=2048,  # 14336
    max_position_embeddings=32768,  # 32768
    num_attention_heads=8,  # 32
    num_hidden_layers=4,  # 32
    num_key_value_heads=2,  # 8
    rms_norm_eps=1e-5,
    rope_theta=10000.0,
    sliding_window=4096,
    tie_word_embeddings=False,
    use_cache=True,
    vocab_size=32000,
    attn_implementation="sdpa",
)



/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


The repository for tiny_shakespeare contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/tiny_shakespeare.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Generating train split:   0%|          | 0/1 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1 [00:00<?, ? examples/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

### Training loop

In [ ]:
from transformers import (
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    PretrainedConfig,
    PreTrainedModel,
)
from torch.utils.data import DataLoader
import torch
import random
import os

def set_seed(seed=42):
    """
    Fix all random seeds we use for reproducibility.
    """
    # Python random seed
    random.seed(seed)

    # PyTorch random seed
    torch.manual_seed(seed)

    # If you are using CUDA
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.

    # PyTorch backend settings
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # Python hash seed
    os.environ["PYTHONHASHSEED"] = str(seed)


def prepare_dataset(tokenizer, dataset):
    def process_batch(batch):
        # Split each text in the batch into lines and strip whitespace
        lines = [line.strip() for text in batch["text"] for line in text.split('\n') if line.strip()]
        # Tokenize the lines
        return tokenizer(lines, padding="max_length", truncation=True, max_length=128, return_tensors="pt")

    # Process the dataset in batches and directly tokenize
    tokenized_dataset = dataset.map(process_batch, batched=True, remove_columns=["text"])
    return tokenized_dataset


def train(with_liger = False):
    lr = 1e-4

    train_dataset = prepare_dataset(tokenizer, dataset)


    if with_liger is True:
        from liger_kernel.transformers import apply_liger_kernel_to_mistral
        apply_liger_kernel_to_mistral(rope=True, rms_norm=True, cross_entropy=True, swiglu=True)

    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    loader = DataLoader(
        train_dataset, batch_size=16, shuffle=False, collate_fn=data_collator
    )

    loader_iter = iter(loader)
    model = MistralForCausalLM(model_config).to("cuda")

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)


    loss_list = []
    num_steps=10


    for i in range(num_steps):
        batch = next(loader_iter).to("cuda")
        output = model(**batch)
        output.loss.backward()
        optimizer.step()
        print(f"Step {i}, Loss: {output.loss.item()}")
        loss_list.append(output.loss.item())

    return {"loss": loss_list, "logits": output.logits, "model": model}


### Fix the random seed, start the training, and compare with and without liger kernel

In [ ]:
set_seed(seed=42)
train_dict_1 = train(with_liger = False)
set_seed(seed=42)
train_dict_2 = train(with_liger = True)

# compare loss
assert torch.allclose(torch.tensor(train_dict_1["loss"]), torch.tensor(train_dict_2["loss"]), atol=1e-8, rtol=1e-5)
# compare logits
assert torch.allclose(train_dict_1["logits"], train_dict_2["logits"], atol=1e-8, rtol=1e-5)
# compare model
for param_1, param_2 in zip(train_dict_1["model"].parameters(), train_dict_2["model"].parameters()):
    assert torch.allclose(param_1, param_2, atol=1e-8, rtol=1e-5)

print("Passed convergence test!")


Step 0, Loss: 10.611056327819824
Step 1, Loss: 9.523656845092773
Step 2, Loss: 8.7361421585083
Step 3, Loss: 8.548308372497559
Step 4, Loss: 8.431111335754395
Step 5, Loss: 8.516386032104492
Step 6, Loss: 8.403151512145996
Step 7, Loss: 7.854078769683838
Step 8, Loss: 8.217671394348145
Step 9, Loss: 7.309558391571045
Step 0, Loss: 10.611056327819824
Step 1, Loss: 9.523656845092773
Step 2, Loss: 8.7361421585083
Step 3, Loss: 8.548308372497559
Step 4, Loss: 8.431111335754395
Step 5, Loss: 8.516386032104492
Step 6, Loss: 8.403151512145996
Step 7, Loss: 7.854078769683838
Step 8, Loss: 8.217671394348145
Step 9, Loss: 7.309558391571045
Passed convergence test!
